# 헬름 사용법

## 차트 생성

- `helm create 차트명`
- `차트명` 디렉토리가 생성되며 그 아래에 기본 템플릿이 만들어짐

```bash
$ helm create helm1
Creating helm1
$ tree helm1       
helm1
├── Chart.yaml
├── charts
├── templates
│   ├── _helpers.tpl
│   ├── deployment.yaml
│   ├── hpa.yaml
│   ├── httproute.yaml
│   ├── ingress.yaml
│   ├── NOTES.txt
│   ├── service.yaml
│   ├── serviceaccount.yaml
│   └── tests
│       └── test-connection.yaml
└── values.yaml

4 directories, 11 files
```

## 챠트 구조

- `Chart.yaml`: 차트 파일
- `values.yaml`: 디폴트 value 저장 파일
- `charts`: 종속성 차트 파일이 저장되는 디렉토리
- `templates`: 차트에 속하는 리소스에 대한 쿠버네티스 매니페스트가 저장되는 디렉토리

### Chart.yaml 파일 내용

```{.yaml}
apiVersion: v2
name: anvil
description: A Helm chart for Kubernetes

# A chart can be either an 'application' or a 'library' chart.
#
# Application charts are a collection of templates that can be packaged into versioned archives
# to be deployed.
#
# Library charts provide useful utilities or functions for the chart developer. They're included as
# a dependency of application charts to inject those utilities and functions into the rendering
# pipeline. Library charts do not define any templates and therefore cannot be deployed.
type: application

# This is the chart version. This version number should be incremented each time you make changes
# to the chart and its templates, including the app version.
# Versions are expected to follow Semantic Versioning (https://semver.org/)
version: 0.1.0

# This is the version number of the application being deployed. This version number should be
# incremented each time you make changes to the application. Versions are not expected to
# follow Semantic Versioning. They should reflect the version the application is using.
# It is recommended to use it with quotes.
appVersion: "1.16.0"
```

- `apiVersion`: `v2` 값은 helm 버전 3 이상을 뜻함
- `type`: `application` 또는 `library`
- `version`: 차트 버전
- `appVersion`: 애플리케이션 버전

### templates 디렉토리

- templates 디렉토리 아래의 `.yaml` 확장자를 가진 파일에 렌더링된 리소스는 쿠버네티스로 생성

### .yaml 파일

- `templates` 디렉토리 아래의 `.yaml` 파일은 쿠버네티스 리소스를 정의하는 템플릿 파일
- `{{ .Values.키경로 }}`: `values.yaml` 파일의 값을 이 위치에 렌더링
- `{{ .Chart.키경로 }}`: `chart.yaml` 파일의 값을 이 위치에 렌더링
- `{{ .Release.키경로 }}`: Release와 관련된 값을 이 위치에 렌더링
  

### .tpl 파일

- `.tpl` 파일에서는 `.yaml` 파일내에서 사용하는 각종 define 블럭을 정의
- define 블럭
  - `{{ define "블럭명" }}` 문장으로 시작
  - `{{ end }}` 문장으로 종료
- `.tpl` 파일의 이름은 관례상 `_`(밑줄)기호로 시작


### include 함수

- .tpl 파일의 define 블럭에서 정의한 내용을 .yaml 파일에 렌더링
- `{{ include 블럭명 컨텍스트 }}`

  - `{{ ... }}` : 앞뒤 공백/개행을 그대로
  - `{{- ... }}` : 앞쪽 공백/개행을 제거
  - `{{ ... -}}` : 뒤쪽 공백/개행을 제거
  - `{{- ... -}}` : 앞뒤 모두 공백/개행을 제거

## 차트 렌더링

- `helm template 차트명 차트디렉토리` 명령으로 생성되는 리소스 매니페스트를 미리 볼 수 있음 

```bash
helm template helm1 .
---
# Source: helm1/templates/serviceaccount.yaml
apiVersion: v1
kind: ServiceAccount
metadata:
  name: helm1
  labels:
    helm.sh/chart: helm1-0.1.0
    app.kubernetes.io/name: helm1
    app.kubernetes.io/instance: helm1
    app.kubernetes.io/version: "1.16.0"
    app.kubernetes.io/managed-by: Helm
automountServiceAccountToken: true
---
# Source: helm1/templates/service.yaml
apiVersion: v1
kind: Service
metadata:
  name: helm1
  labels:
    helm.sh/chart: helm1-0.1.0
    app.kubernetes.io/name: helm1
    app.kubernetes.io/instance: helm1
    app.kubernetes.io/version: "1.16.0"
    app.kubernetes.io/managed-by: Helm
spec:
  type: ClusterIP
  ports:
    - port: 80
      targetPort: http
      protocol: TCP
      name: http
  selector:
    app.kubernetes.io/name: helm1
    app.kubernetes.io/instance: helm1
---
# Source: helm1/templates/deployment.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: helm1
  labels:
    helm.sh/chart: helm1-0.1.0
    app.kubernetes.io/name: helm1
    app.kubernetes.io/instance: helm1
    app.kubernetes.io/version: "1.16.0"
    app.kubernetes.io/managed-by: Helm
spec:
  replicas: 1
  selector:
    matchLabels:
      app.kubernetes.io/name: helm1
      app.kubernetes.io/instance: helm1
  template:
    metadata:
      labels:
        helm.sh/chart: helm1-0.1.0
        app.kubernetes.io/name: helm1
        app.kubernetes.io/instance: helm1
        app.kubernetes.io/version: "1.16.0"
        app.kubernetes.io/managed-by: Helm
    spec:
      serviceAccountName: helm1
      containers:
        - name: helm1
          image: "nginx:1.16.0"
          imagePullPolicy: IfNotPresent
          ports:
            - name: http
              containerPort: 80
              protocol: TCP
          livenessProbe:
            httpGet:
              path: /
              port: http
          readinessProbe:
            httpGet:
              path: /
              port: http
---
# Source: helm1/templates/tests/test-connection.yaml
apiVersion: v1
kind: Pod
metadata:
  name: "helm1-test-connection"
  labels:
    helm.sh/chart: helm1-0.1.0
    app.kubernetes.io/name: helm1
    app.kubernetes.io/instance: helm1
    app.kubernetes.io/version: "1.16.0"
    app.kubernetes.io/managed-by: Helm
  annotations:
    "helm.sh/hook": test
spec:
  containers:
    - name: wget
      image: busybox
      command: ['wget']
      args: ['helm1:80']
  restartPolicy: Never
```

## 헬름 앱 설치

- helm 차트로 만들어진 앱을 쿠버네티스에 설치하려면 `helm install` 명령을 사용한다.
- `helm install 앱_릴리스_이름 챠트_디렉토리`
- "앱_릴리스_이름"은 차트 이름과 달라도 되다.

```bash
$ ls 
helm1

$ helm install helm1a helm1
NAME: helm1a
LAST DEPLOYED: Mon Jun  1 14:58:14 2026
NAMESPACE: default
STATUS: deployed
REVISION: 1
DESCRIPTION: Install complete
NOTES:
1. Get the application URL by running these commands:
  export POD_NAME=$(kubectl get pods --namespace default -l "app.kubernetes.io/name=helm1,app.kubernetes.io/instance=helm1a" -o jsonpath="{.items[0].metadata.name}")
  export CONTAINER_PORT=$(kubectl get pod --namespace default $POD_NAME -o jsonpath="{.spec.containers[0].ports[0].containerPort}")
  echo "Visit http://127.0.0.1:8080 to use your application"
  kubectl --namespace default port-forward $POD_NAME 8080:$CONTAINER_PORT

$ kubectl get sa,svc,deploy,pod  -l app.kubernetes.io/name=helm1
NAME                    SECRETS   AGE
serviceaccount/helm1a   0         14m

NAME             TYPE        CLUSTER-IP     EXTERNAL-IP   PORT(S)   AGE
service/helm1a   ClusterIP   10.43.85.177   <none>        80/TCP    14m

NAME                     READY   UP-TO-DATE   AVAILABLE   AGE
deployment.apps/helm1a   1/1     1            1           14m

NAME                          READY   STATUS    RESTARTS   AGE
pod/helm1a-6777c58455-mgcc8   1/1     Running   0          14m
```

## 헬름 앱 삭제

- helm 차트로 만들어지고 쿠버네티스에 설치된 앱을 쿠버네티스에서 삭제하려면 `helm uninstall` 명령을 사용한다.
- `helm install 앱_릴리스_이름`